### **Laboratorio 11: coordinación, fallos y HotpotQA real**


Trabajaremos primero sobre las 7.405 preguntas originales. Después ejecutaremos pruebas controladas de contratos y analizaremos un piloto LLM solo si los recursos de Docker lo permiten.

Entrega un informe breve con IDs de consultas reales, métricas reproducibles, trazas y amenazas a la validez. No fabriques respuestas ni casos fallidos que no aparecieron.

#### **1. Entorno y datos verificables**

Ejecuta este laboratorio desde `/workspace/CC-0F4` con HotpotQA preparado previamente. Si falta el archivo o el hash no coincide, corrige la descarga. No uses datos sustitutivos para obtener un resultado positivo.

In [1]:
from pathlib import Path
import os, json, sys, csv, collections, copy
root = Path.cwd().resolve()
for candidate in (root, *root.parents):
    if (candidate / 'Semana11/multiagente_real.py').is_file():
        root = candidate
        break
else:
    raise FileNotFoundError('Abre JupyterLab dentro de CC-0F4')
if str(root) not in sys.path:
    sys.path.insert(0,str(root))
from Semana11.multiagente_real import (
    check_dataset, ranked_evidence, gold_facts, support_recall, retrieval_rows,
    summarize_retrieval, evaluate_retrieval, validate_answer, select_pilot, pilot, MODEL_ID)
from Semana11.contratos import Blackboard, ContractError, make_message, validate_selection
examples = check_dataset()
assert len(examples) == 7405
print('HotpotQA verificado:', len(examples))

HotpotQA verificado: 7405


#### **2. Lectura de un problema real**

Selecciona una pregunta directamente del dataset. Anota su identificador, el tipo de pregunta, sus párrafos disponibles y la estructura de las anotaciones humanas. El código no necesita crear preguntas nuevas.

In [2]:
ex = examples[0]
print('ID:',ex['_id'])
print('Pregunta:',ex['question'])
print('Tipo:',ex['type'],'| nivel:',ex['level'])
print('Párrafos disponibles:',len(ex['context']))
print('Oraciones de apoyo:',len(ex['supporting_facts']))

ID: 5a8b57f25542995d1e6f1371
Pregunta: Were Scott Derrickson and Ed Wood of the same nationality?
Tipo: comparison | nivel: hard
Párrafos disponibles: 10
Oraciones de apoyo: 2


#### **3. Ranking y trazabilidad**

Calcula top-14 con BM25, registra título, índice de oración y puntuación. Justifica por qué el recuperador no debe recibir `supporting_facts`. Repite la recuperación y verifica que el orden sea idéntico.

In [3]:
observed = ranked_evidence(ex,14)
assert observed == ranked_evidence(ex,14)
for item in observed[:7]:
    print(item['id'], item['title'], item['sent_id'],
          f"BM25={item['bm25']:.4f}",item['text'][:115])

E00 Woodson, Arkansas 4 BM25=6.3390  Woodson is adjacent to the Wood Plantation, the largest of the plantations own by Ed Wood Sr.
E01 Woodson, Arkansas 3 BM25=5.8976  Woodson and its accompanying Woodson Lake and Wood Hollow are the namesake for Ed Wood Sr., a prominent plantation
E02 Ed Wood (film) 0 BM25=5.1273 Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring 
E03 Conrad Brooks 3 BM25=4.9739  He took a break from acting during the 1960s and 1970s but due to the ongoing interest in the films of Ed Wood, he
E04 Sinister (film) 0 BM25=4.9450 Sinister is a 2012 supernatural horror film directed by Scott Derrickson and written by Derrickson and C. Robert Ca
E05 Scott Derrickson 0 BM25=4.6869 Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.
E06 Doctor Strange (2016 film) 0 BM25=4.6387 Doctor Strange is a 2016 American superhero film based on the Marvel Comics character of the same na

#### **4. Métricas de evidencia, no de respuesta**

Calcula cuántos hechos gold aparecen entre las oraciones seleccionadas. Compara AnyHit y AllHit. Responde por qué una pregunta con AnyHit uno todavía puede no tener toda la evidencia de varios saltos.

In [4]:
gold = gold_facts(ex)
found = {(i['title'],i['sent_id']) for i in observed}
print('Gold:',len(gold),'intersección:',len(gold & found))
print('Recall:',support_recall(ex,observed))
print('AnyHit:',bool(gold & found),'AllHit:',bool(gold and gold <= found))

Gold: 2 intersección: 1
Recall: 0.5
AnyHit: True AllHit: False


#### **5. Evaluación obligatoria de las 7.405 preguntas**

Se reutilizan el resumen y las trazas reales si ambos existen. Si no existe ninguno, se ejecuta por primera vez el benchmark de 7.405 preguntas. No se sustituyen datos ni se generan resultados artificiales. Si solo existe uno de los archivos, se debe recuperar la evidencia faltante antes de continuar.

In [ ]:
results_dir = root / 'Semana11' / 'resultados'
summary_path = results_dir / 'hotpot_recuperacion_resumen.json'
rows_path = results_dir / 'hotpot_recuperacion_7405_por_consulta.jsonl'
if summary_path.is_file() and rows_path.is_file():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('Resultados de recuperación almacenados. No se repite el benchmark.')
elif not summary_path.exists() and not rows_path.exists():
    summary = evaluate_retrieval()
    print('Primera evaluación de recuperación.')
else:
    raise FileNotFoundError('Evidencia incompleta: restaura ambos archivos de recuperación antes de continuar.')
assert summary['questions'] == 7405
assert summary['top_k'] == 14
assert summary['source_sha256'] == '4e9ecb5c8d3b719f624d66b60f8d56bf227f03914f5f0753d6fa1b359d7104ea'
print(json.dumps(summary['retrieval'], ensure_ascii=False, indent=2))
rows = [json.loads(line) for line in rows_path.read_text(encoding='utf-8').splitlines()]
assert len(rows) == 7405
print('Trazabilidad:', len(rows), 'filas')


#### **6. Fallos empíricos de recuperación**

Busca preguntas con AllHit igual a cero, registrando sus IDs, tipo y número de hechos gold. No uses preguntas inventadas. Si no existe un grupo solicitado, indica que no aparece en esta ejecución.

In [6]:
failures = [row for row in rows if row['all_support'] == 0]
print('Preguntas sin recuperación completa:',len(failures))
for row in failures[:5]:
    print(row['question_id'],row['type'],row['level'],
          'gold',row['n_gold'],'hallados',row['n_support_found'])

Preguntas sin recuperación completa: 2823
5a8b57f25542995d1e6f1371 comparison hard gold 2 hallados 1
5a8c7595554299585d9e36b6 bridge hard gold 3 hallados 1
5a85ea095542994775f606a8 bridge hard gold 5 hallados 3
5a87ab905542996e4f3088c1 bridge hard gold 2 hallados 1
5a8db19d5542994ba4e3dd00 comparison hard gold 2 hallados 1


#### **7. Comparación por tipo de pregunta**

Contrasta `bridge` y `comparison` si ambos aparecen. Explica por qué el tipo de pregunta puede modificar la necesidad de dos fragmentos de evidencia. No atribuyas automáticamente estas diferencias a la arquitectura multiagente.

In [7]:
for typ,values in summary['by_type'].items():
    print(typ,'N=',values['questions'],
          'AnyHit=',round(values['any_support_rate'],4),
          'AllHit=',round(values['all_support_rate'],4))

bridge N= 5918 AnyHit= 0.9816 AllHit= 0.5874
comparison N= 1487 AnyHit= 0.9825 AllHit= 0.7438


#### **8. Contrato de mensajes y pizarra**

Construye un mensaje a partir de los identificadores realmente recuperados y verifica que el supervisor lo acepta. Comprueba que cambiar después el objeto original no modifica el contenido ya registrado.

In [8]:
board=Blackboard(ex['_id'])
ids=[observed[0]['id'], observed[1]['id']]
message=make_message(ex['_id'],'evidence_worker',ids,observed)
board.append(message,observed)
message['selected_ids'].clear()
print('Pizarra validada:',board.messages[0]['selected_ids'])
assert board.messages[0]['selected_ids']==ids

Pizarra validada: ['E00', 'E01']


#### **9. Fallos de ingeniería inyectados**

Los siguientes errores son pruebas de contratos y deben mantenerse fuera de las métricas de HotpotQA. El objetivo es explicar el mecanismo de rechazo, no simular un agente real que falla.

In [9]:
# Se registran motivos de rechazo en los contratos, no métricas sintéticas.
cases = [
    ('cita_no_observada', lambda: validate_selection(['E99'],observed)),
    ('citas_duplicadas', lambda: validate_selection(['E00','E00'],observed)),
    ('seleccion_vacia', lambda: validate_selection([],observed)),
    ('rol_no_autorizado', lambda: make_message(ex['_id'],'admin',['E00'],observed)),
]
for label, func in cases:
    try:
        func()
    except ContractError as err:
        print(label, 'rechazado:',err)
    else:
        raise AssertionError('Se aceptó el fallo '+label)

cita_no_observada rechazado: CITA_NO_OBSERVADA
citas_duplicadas rechazado: ID_DUPLICADO
seleccion_vacia rechazado: SELECCION_VACIA_O_EXCESIVA
rol_no_autorizado rechazado: ROL_NO_AUTORIZADO


#### **10. Procedencia y huellas de texto**

La comprobación de identificadores puede ser insuficiente si alguien modifica un fragmento después de la recuperación. La huella SHA-256 debe cubrir contenido y localizador. Comprueba el rechazo de una huella alterada.

In [10]:
invalid=copy.deepcopy(board.messages[0])
invalid['evidence_sha256'][ids[0]]='hash_incorrecto'
try:
    board.append(invalid,observed)
except ContractError as err:
    print('Procedencia alterada detectada:',err)
else:
    raise AssertionError('El contrato aceptó un hash falso')

Procedencia alterada detectada: PROCEDENCIA_ALTERADA


#### **11. Contrato de respuesta final**

Distingue una finalización estructural válida de una respuesta semánticamente correcta. La prueba utiliza el formato del contrato, no una afirmación sobre la respuesta humana a la pregunta.

In [11]:
ok,reason,_,cites=validate_answer({'answer':'Revisar evidencia original',
                                           'cited_ids':[ids[0]]},observed)
print(ok,reason,cites)
assert ok
invalid,invalid_reason,_,_=validate_answer({'answer':'texto','cited_ids':['E99']},observed)
print(invalid,invalid_reason)
assert not invalid

True FINALIZACION_ESTRUCTURAL ['E00']
False CITA_NO_OBSERVADA


#### **12. Pruebas unitarias del controlador**

Ejecuta los tests sin red ni Qwen. Las entradas pequeñas de los tests sirven para verificar contratos; nunca se mezclan con las 7.405 preguntas auténticas que componen el benchmark.

In [12]:
import subprocess
result=subprocess.run([sys.executable,'-m','unittest','discover','-s',
                       'Semana11/tests','-t','.','-p','test_*.py','-v'],
                      cwd=root,capture_output=True,text=True)
print(result.stdout[-3000:]); print(result.stderr[-3000:])
assert result.returncode==0


ts_duplicate_citations) ... ok
test_answer_requires_observed_citation (Semana11.tests.test_multiagente_real.CoordinationTests.test_answer_requires_observed_citation) ... ok
test_blackboard_preserves_provenance (Semana11.tests.test_multiagente_real.CoordinationTests.test_blackboard_preserves_provenance) ... ok
test_blackboard_rejects_other_question (Semana11.tests.test_multiagente_real.CoordinationTests.test_blackboard_rejects_other_question) ... ok
test_blackboard_rejects_tampered_content (Semana11.tests.test_multiagente_real.CoordinationTests.test_blackboard_rejects_tampered_content) ... ok
test_blackboard_rejects_unknown_role (Semana11.tests.test_multiagente_real.CoordinationTests.test_blackboard_rejects_unknown_role) ... ok
test_empty_context_rejected (Semana11.tests.test_multiagente_real.CoordinationTests.test_empty_context_rejected) ... ok
test_evidence_digest_detects_alteration (Semana11.tests.test_multiagente_real.CoordinationTests.test_evidence_digest_detects_alteration) ... o

#### **13. Piloto opcional con inferencia LLM real**

Por defecto se consultan los resultados reales del piloto de 16 preguntas y sus trazas, si están disponibles. No se descarga Qwen ni se ejecutan inferencias al consultar esos archivos.

 Solo se realiza una nueva ejecución si se activa explícitamente `CC0F4_RUN_REAL_LLM=1`. Las variables `CC0F4_S11_LLM_N` y `CC0F4_S11_MODEL` configuran una nueva ejecución, no cambian los resultados almacenados. Una nueva ejecución puede sobrescribir sus archivos de salida, por lo que se debe respaldar primero la evidencia previa. Las finalizaciones inválidas se registran sin repararlas.

In [ ]:
results_dir = root / 'Semana11' / 'resultados'
pilot_path = results_dir / 'hotpot_piloto_16_resumen.json'
traces_path = results_dir / 'hotpot_piloto_16_trazas.jsonl'
if os.getenv('CC0F4_RUN_REAL_LLM', '0') == '1':
    count = int(os.getenv('CC0F4_S11_LLM_N', '16'))
    report = pilot(count, os.getenv('CC0F4_S11_MODEL', MODEL_ID))
    print('Nueva ejecución experimental solicitada explícitamente.')
    print(json.dumps(report, ensure_ascii=False, indent=2))
elif pilot_path.is_file() and traces_path.is_file():
    report = json.loads(pilot_path.read_text(encoding='utf-8'))
    assert report['questions'] == 16
    assert report['dataset_sha256'] == '4e9ecb5c8d3b719f624d66b60f8d56bf227f03914f5f0753d6fa1b359d7104ea'
    assert report['status'] == 'PILOTO_REAL_EXPLORATORIO'
    print('Piloto real almacenado. No se realizan nuevas inferencias.')
    print(json.dumps(report, ensure_ascii=False, indent=2))
    print('Trazas:', traces_path)
elif pilot_path.exists() or traces_path.exists():
    raise FileNotFoundError('Evidencia del piloto incompleta: restaura el resumen y las trazas.')
else:
    report = None
    print('No hay resultados del piloto de 16 preguntas. No se realizaron inferencias en esta sesión.')


#### **14. Análisis de resultados y defensa**

En tu informe indica: el número real de preguntas, AllHit y Recall medios, al menos dos IDs de error del benchmark, la razón de rechazo de tres mensajes malformados y una limitación de la comparación multiagente. Si analizaste el piloto almacenado o ejecutaste uno nuevo, aporta EM, F1, tasas de finalización válida, tokens y fallos por condición.

No uses una finalización estructural válida como prueba de factualidad. No prometas que un supervisor con dos trabajadores supera a un agente único cuando la evidencia no lo demuestra.

#### **15. Entrega**

Adjunta el notebook ejecutado, el resumen de recuperación, una selección de trazas reales de error, el resultado de las pruebas unitarias y un comentario crítico de hasta dos páginas. Distingue el análisis del piloto almacenado de una nueva ejecución. Si no existe evidencia de inferencia LLM real, indica que el piloto queda pendiente.

Identifica las rutas y los hashes de datos. No cambies preguntas ni respuestas humanas para producir métricas favorables.